# NetGuardAI — Module 1: Data Ingestion and Preprocessing

**Project:** NetGuardAI | FYP-1 | FAST-NUCES  
**Dataset:** GeNIS — GECAD Network Intrusion Scenarios (2025)  
**Zenodo:** https://zenodo.org/records/14919237  
**Citation:** https://doi.org/10.1016/j.dib.2025.111487  

---

### Module 1 Task Breakdown (Leak-Free Scientific Pipeline)

| Task | Name | Scientific Requirement |
|------|------|------------------------|
| T1.1 | Dataset Acquisition | Stream `4-preprocessed.zip` from Zenodo |
| T1.2 | Data Loading & Scenario Assembly | Load both 60s train and test partitions to maintain complete scenario coverage |
| T1.3 | Exploratory Data Analysis (EDA) | 4 comprehensive diagnostic charts (distributions, correlations, high-variance histograms) |
| T1.4 | Target Isolation & Stratified Split | **Split FIRST (70/15/15)**; eliminate target leakage (`CategoryLabel` target; isolate `BinaryLabel`/`SubCategoryLabel`) |
| T1.5 | Feature Engineering (Train-Fitted) | Impute medians and encode categoricals using **Training Split statistics only** |
| T1.6 | Normalization (Train-Fitted) | Fit `MinMaxScaler` on **Training Split only**; transform validation and test splits |
| T1.7 | Class Imbalance Report | Quantify $N_{majority} : N_{minority}$ ratio to formally justify CTGAN augmentation |
| T1.8 | Baseline Benchmark (No Augmentation) | Train Random Forest on leak-free baseline data to document true minority detection failures |


---
## Setup — Install and Import All Libraries

In [ ]:
# Install required libraries (only needed in Google Colab)
!pip install -q pandas numpy matplotlib seaborn scikit-learn requests

In [ ]:
import os
import json
import zipfile
import warnings
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

# Global plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Colab vs Local directory handling
BASE_DIR = '/content' if os.path.exists('/content') else '.'
EXTRACT_DIR = os.path.join(BASE_DIR, 'genis_data')
ZIP_PATH = os.path.join(BASE_DIR, 'genis_preprocessed.zip')

print('All libraries imported successfully.')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Working directory: {os.path.abspath(BASE_DIR)}')

---
## T1.1 — Dataset Acquisition

We download the **preprocessed CSV variant** (`4-preprocessed.zip`, ~600 MB) from Zenodo Record 14919237.  
This variant provides filtered network flow features aggregated over 5s, 10s, 30s, and 60s intervals.

In [ ]:
# Zenodo direct download URL — preprocessed CSV variant
ZENODO_URL = 'https://zenodo.org/records/14919237/files/4-preprocessed.zip?download=1'

os.makedirs(EXTRACT_DIR, exist_ok=True)

def download_file(url, save_path):
    """Download file with chunked streaming to prevent memory pressure."""
    if os.path.exists(save_path) and os.path.getsize(save_path) > 10 * 1024 * 1024:
        print(f'Archive already downloaded: {save_path} ({os.path.getsize(save_path)/(1024*1024):.1f} MB)')
        return
    print('Downloading 4-preprocessed.zip from Zenodo...')
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total = int(response.headers.get('content-length', 0))
    downloaded = 0
    chunk_size = 1024 * 1024  # 1 MB

    with open(save_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f'  Progress: {pct:.1f}% ({downloaded // (1024*1024)} MB / {total // (1024*1024)} MB)', end='\r')
    print()
    print(f'Download complete. Size: {os.path.getsize(save_path)/(1024*1024):.1f} MB')

download_file(ZENODO_URL, ZIP_PATH)

# Extract ZIP
print('Extracting ZIP archive...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    contents = z.namelist()
    print(f'Files inside archive ({len(contents)} total):')
    for item in contents[:10]:
        print(f'  - {item}')
    if len(contents) > 10:
        print(f'  ... and {len(contents)-10} more files')
    z.extractall(EXTRACT_DIR)
print(f'Extraction complete to: {EXTRACT_DIR}')

---
## T1.2 — Data Loading and Scenario Assembly

GeNIS provides 60-second aggregated flows. To ensure complete scenario coverage and prevent
accidentally taking only the benchmark test partition, we identify all 60s files
(both `genis-60-sec-train.csv` and `genis-60-sec-test.csv`) and assemble the full dataset.

In [ ]:
# Locate all CSV files
csv_files = sorted(Path(EXTRACT_DIR).rglob('*.csv'))
print(f'Found {len(csv_files)} CSV file(s) in {EXTRACT_DIR}:')
for i, f in enumerate(csv_files):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'  [{i}] {f.name} ({size_mb:.1f} MB)')

# Locate 60-second window files (both train and test partitions)
files_60s = [f for f in csv_files if '60' in f.name]

if len(files_60s) == 0:
    print('\nNo 60s files found. Falling back to the largest CSV file.')
    selected_files = [max(csv_files, key=lambda f: f.stat().st_size)]
else:
    selected_files = files_60s
    print(f'\nSelected {len(selected_files)} 60-second interval file(s):')
    for f in selected_files:
        print(f'  -> {f.name}')

# Load and concatenate all 60s scenario partitions to form complete master dataset
dfs = []
for f in selected_files:
    print(f'Loading: {f.name} ...')
    part = pd.read_csv(f, low_memory=False)
    dfs.append(part)

df = pd.concat(dfs, ignore_index=True)
print(f'\nMaster Dataset Assembled Successfully.')
print(f'Shape : {df.shape[0]:,} rows x {df.shape[1]} columns')

In [ ]:
# Dataset Inspection Report
separator = '=' * 65
print(separator)
print('NETGUARDAI — DATASET INSPECTION REPORT')
print('GeNIS (2025)  |  Module 1 — T1.2')
print(separator)
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')

mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f'Memory  : {mem_mb:.2f} MB')

# Check GeNIS multi-tier label columns
KNOWN_LABELS = ['BinaryLabel', 'CategoryLabel', 'SubCategoryLabel']
found_labels = [c for c in KNOWN_LABELS if c in df.columns]
print(f'\nGeNIS Label Tiers Detected: {found_labels}')
for lbl in found_labels:
    print(f'  [{lbl}] Unique values: {df[lbl].nunique()}')
    print(f'    Distribution:\n{df[lbl].value_counts().head(5).to_dict()}')

print(f'\nMISSING VALUES')
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print('  No missing values found.')
else:
    print(f'  {len(missing_cols)} column(s) have missing values:')
    for col, cnt in missing_cols.items():
        print(f'    {col}: {cnt:,} ({cnt/len(df)*100:.2f}%)')

print(f'\nDUPLICATE ROWS (exact full-row matches)')
dupes = df.duplicated().sum()
print(f'  {dupes:,} duplicate rows ({dupes/len(df)*100:.3f}%)')
print(separator)

---
## T1.3 — Exploratory Data Analysis (EDA)

We produce 4 diagnostic charts:
1. **Class Distribution Bar Chart** (with log scale for minority visibility)
2. **Missing Values Matrix**
3. **Feature Correlation Matrix** (Top 20 numeric features)
4. **Feature Distributions** (Top 9 high-variance traffic features)

In [ ]:
# Select primary target column for analysis
# Primary target: CategoryLabel (Multiclass attack types: Benign, Bruteforce, DoS, Reconnaissance)
if 'CategoryLabel' in df.columns:
    PRIMARY_LABEL = 'CategoryLabel'
elif 'BinaryLabel' in df.columns:
    PRIMARY_LABEL = 'BinaryLabel'
else:
    PRIMARY_LABEL = df.columns[-1]

print(f'Primary Analysis Target: "{PRIMARY_LABEL}"')

# Chart 1 — Class Distribution (Dual View: Linear and Log Scale)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
class_counts = df[PRIMARY_LABEL].value_counts()
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(class_counts)))

# Linear Scale
bars1 = ax1.bar(class_counts.index.astype(str), class_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars1, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + class_counts.max()*0.01,
             f'{val:,}', ha='center', va='bottom', fontsize=8)
ax1.set_title(f'Class Distribution (Linear Scale)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Flow Count')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.tick_params(axis='x', rotation=30)

# Log Scale (Critical for visualizing rare attack classes)
bars2 = ax2.bar(class_counts.index.astype(str), class_counts.values, color=colors, edgecolor='white')
ax2.set_yscale('log')
ax2.set_title(f'Class Distribution (Log Scale — Rare Classes Visible)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Flow Count (Log Scale)')
ax2.tick_params(axis='x', rotation=30)

plt.suptitle(f'GeNIS Dataset — {PRIMARY_LABEL} Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
chart1_path = os.path.join(BASE_DIR, 'chart1_class_distribution.png')
plt.savefig(chart1_path, bbox_inches='tight')
plt.show()
print(f'Chart 1 saved: {chart1_path}')

In [ ]:
# Chart 2 — Missing Values Heatmap
missing_pct = df.isnull().mean() * 100
missing_df = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(missing_df) == 0:
    print('No missing values in the dataset. Missing values heatmap not required.')
else:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.4)))
    missing_matrix = df[missing_df.index].isnull().astype(int)
    sns.heatmap(missing_matrix.T, cbar=False, cmap='Blues', ax=ax, yticklabels=True)
    ax.set_title('Missing Values Heatmap', fontsize=13, fontweight='bold')
    ax.set_xlabel('Sample Index')
    plt.tight_layout()
    chart2_path = os.path.join(BASE_DIR, 'chart2_missing_values.png')
    plt.savefig(chart2_path, bbox_inches='tight')
    plt.show()
    print(f'Chart 2 saved: {chart2_path}')

In [ ]:
# Chart 3 — Feature Correlation Matrix (Top 20 numeric features by variance)
# Exclude label columns before calculating correlation
label_cols_all = ['BinaryLabel', 'CategoryLabel', 'SubCategoryLabel']
num_candidates = [c for c in df.select_dtypes(include=[np.number]).columns if c not in label_cols_all]

top_cols = df[num_candidates].var().nlargest(20).index.tolist()
corr_matrix = df[top_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', center=0, linewidths=0.3, ax=ax)
ax.set_title('Feature Correlation Matrix (Top 20 High-Variance Traffic Features)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
chart3_path = os.path.join(BASE_DIR, 'chart3_correlation_matrix.png')
plt.savefig(chart3_path, bbox_inches='tight')
plt.show()
print(f'Chart 3 saved: {chart3_path}')

In [ ]:
# Chart 4 — Feature Distribution (Top 9 high-variance numeric features)
top9 = df[num_candidates].var().nlargest(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(top9):
    data = df[col].dropna()
    # Clip extreme outliers to 99th percentile for visual clarity
    q99 = data.quantile(0.99)
    clipped = data[data <= (q99 if q99 > 0 else data.max())]
    axes[i].hist(clipped, bins=40, color='steelblue', edgecolor='none', alpha=0.85)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=7)
    axes[i].set_ylabel('Frequency', fontsize=7)
    axes[i].tick_params(labelsize=7)

fig.suptitle('Feature Distributions — Top 9 High-Variance Traffic Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
chart4_path = os.path.join(BASE_DIR, 'chart4_feature_distributions.png')
plt.savefig(chart4_path, bbox_inches='tight')
plt.show()
print(f'Chart 4 saved: {chart4_path}')

---
## T1.4 — Target Isolation & Stratified Splitting (Split FIRST)

### Key Methodological Safeguards:
1. **Target Leakage Elimination:**  
   We set `CategoryLabel` as our target `y`. All other label columns (`BinaryLabel`, `SubCategoryLabel`)
   are **strictly removed** from feature matrix `X` so the model cannot trivially infer the answer.
2. **Preserving Network Traffic Statistics:**  
   We drop only true host/session identifiers (IPs, MACs, timestamps). We **strictly preserve** flow statistics:
   `SrcPkts`, `DstPkts`, `SrcBytes`, `DstBytes`, `SrcRate`, `DstRate`, `sport`, `dport`.
3. **Split FIRST (70% Train, 15% Validation, 15% Test):**  
   The dataset is split **before** any imputers, encoders, or scalers are fitted.

In [ ]:
# Step 1 — Define Target and Isolate All Label Tiers
ALL_LABEL_COLS = ['BinaryLabel', 'CategoryLabel', 'SubCategoryLabel']
TARGET_COL = 'CategoryLabel' if 'CategoryLabel' in df.columns else 'BinaryLabel'

print(f'Selected Target Column : "{TARGET_COL}"')
labels_to_exclude = [c for c in ALL_LABEL_COLS if c in df.columns]
print(f'Label Columns Excluded from Features (Target Leakage Prevention): {labels_to_exclude}')

# Separate features X and target y
X_raw = df.drop(columns=labels_to_exclude).copy()
y_raw = df[TARGET_COL].copy()

# Step 2 — Refined Identifier Removal (Preserve Ports and Traffic Features)
# Drop ONLY true identifiers (IPs, MACs, timestamps, session keys)
id_keywords = ['saddr', 'daddr', 'src_ip', 'dst_ip', 'mac', 'smac', 'dmac', 'flow_id', 'record_id', 'timestamp', 'stime', 'ltime']
cols_to_drop = [c for c in X_raw.columns if any(kw == c.lower() or kw in c.lower() for kw in id_keywords)]

# Crucial Check: Ensure traffic statistics (Pkts, Bytes, Rate, Ports) are NOT accidentally dropped
preserve_keywords = ['pkt', 'byte', 'rate', 'load', 'loss', 'port', 'sport', 'dport', 'dur', 'state', 'proto']
cols_to_drop = [c for c in cols_to_drop if not any(p in c.lower() for p in preserve_keywords)]

print(f'\nIdentifier Columns Dropped: {len(cols_to_drop)}')
for c in cols_to_drop:
    print(f'  - {c}')
X_raw.drop(columns=cols_to_drop, inplace=True)

# Step 3 — Drop Constant (Zero-Variance) Columns
constant_cols = [c for c in X_raw.columns if X_raw[c].nunique() <= 1]
print(f'\nConstant Columns Dropped: {len(constant_cols)}')
for c in constant_cols:
    print(f'  - {c}')
X_raw.drop(columns=constant_cols, inplace=True)

# Step 4 — Encode Target Labels
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y_raw.astype(str))
class_names = le_target.classes_
print(f'\nTarget Classes ({len(class_names)}): {list(class_names)}')

# Step 5 — Stratified Train / Validation / Test Split (70 / 15 / 15)
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=42
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f'\nStratified Split Complete:')
print(f'  Train set      : {X_train_raw.shape[0]:>8,} rows ({X_train_raw.shape[0]/len(X_raw)*100:.1f}%)')
print(f'  Validation set : {X_val_raw.shape[0]:>8,} rows ({X_val_raw.shape[0]/len(X_raw)*100:.1f}%)')
print(f'  Test set       : {X_test_raw.shape[0]:>8,} rows ({X_test_raw.shape[0]/len(X_raw)*100:.1f}%)')

---
## T1.5 — Feature Engineering (Training Split Only)

Missing value imputers and categorical encoders are **fitted strictly on `X_train_raw`**,
then transformed across validation and test sets without re-fitting. This prevents data leakage.

In [ ]:
numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols     = X_train_raw.select_dtypes(include=['object', 'category']).columns.tolist()

X_train_clean = X_train_raw.copy()
X_val_clean   = X_val_raw.copy()
X_test_clean  = X_test_raw.copy()

# 1. Numeric Imputation — Calculate medians on TRAINING split only
train_medians = X_train_raw[numeric_cols].median()
X_train_clean[numeric_cols] = X_train_clean[numeric_cols].fillna(train_medians)
X_val_clean[numeric_cols]   = X_val_clean[numeric_cols].fillna(train_medians)
X_test_clean[numeric_cols]  = X_test_clean[numeric_cols].fillna(train_medians)

# 2. Categorical Imputation and Encoding — Fitted on TRAINING split only
cat_encoders = {}
for col in cat_cols:
    train_mode = X_train_raw[col].mode()[0] if not X_train_raw[col].empty else 'missing'
    X_train_clean[col].fillna(train_mode, inplace=True)
    X_val_clean[col].fillna(train_mode, inplace=True)
    X_test_clean[col].fillna(train_mode, inplace=True)

    le = LabelEncoder()
    X_train_clean[col] = le.fit_transform(X_train_clean[col].astype(str))
    cat_encoders[col] = le

    # Safe transform for Val/Test handling any unseen categories
    known_classes = set(le.classes_)
    default_class = le.classes_[0]
    X_val_clean[col]  = X_val_clean[col].map(lambda x: x if x in known_classes else default_class)
    X_test_clean[col] = X_test_clean[col].map(lambda x: x if x in known_classes else default_class)
    X_val_clean[col]  = le.transform(X_val_clean[col].astype(str))
    X_test_clean[col] = le.transform(X_test_clean[col].astype(str))

print('Feature Engineering complete (Fitted on Training Set Only):')
print(f'  Numeric features imputed : {len(numeric_cols)}')
print(f'  Categorical features encoded : {len(cat_cols)}')
print(f'  Remaining missing in Train : {X_train_clean.isnull().sum().sum()}')
print(f'  Remaining missing in Val   : {X_val_clean.isnull().sum().sum()}')
print(f'  Remaining missing in Test  : {X_test_clean.isnull().sum().sum()}')

---
## T1.6 — Normalization (Training Split Only)

We fit `MinMaxScaler` **only on the training split** to establish scale bounds $[0, 1]$.
Validation and test sets are transformed using the fitted scaler, with no re-fitting.

In [ ]:
feature_cols = X_train_clean.columns.tolist()

# Fit scaler on training split ONLY
scaler = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_clean), columns=feature_cols)
X_val   = pd.DataFrame(scaler.transform(X_val_clean),       columns=feature_cols)
X_test  = pd.DataFrame(scaler.transform(X_test_clean),      columns=feature_cols)

print('Min-Max Normalization applied strictly without data leakage.')
print(f'X_train shape : {X_train.shape}')
print(f'X_val shape   : {X_val.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'Train bounds  : min={X_train.min().min():.4f}, max={X_train.max().max():.4f}')
print(f'Test range    : min={X_test.min().min():.4f}, max={X_test.max().max():.4f}')

---
## T1.7 — Class Imbalance Report

This report formally quantifies the class imbalance in the training partition.
We identify majority and minority classes, compute the empirical imbalance ratio,
and officially justify CTGAN synthetic data augmentation in Module 2.

In [ ]:
# Class Imbalance Analysis on Training Set
counts_series = pd.Series(y_train).value_counts().sort_values(ascending=False)
class_idx_list = counts_series.index.tolist()
counts = counts_series.values
total = counts.sum()
names = [class_names[i] for i in class_idx_list]

separator = '=' * 75
print(separator)
print('NETGUARDAI — CLASS IMBALANCE REPORT')
print('GeNIS Training Partition  |  Module 1 — T1.7')
print(separator)
print(f'{"Rank":<5} {"Class":<25} {"Count":>10} {"Percentage":>12}  {"Status"}')
print('-' * 75)

# Dynamic definition: Majority is the top class; all other classes are minority
minority_info = []
majority_info = (names[0], counts[0], counts[0]/total*100)

for rank, (cls_name, cnt) in enumerate(zip(names, counts), 1):
    pct = cnt / total * 100
    if rank == 1:
        status = 'MAJORITY (Baseline Traffic)'
    else:
        status = 'MINORITY (Underrepresented Attack)'
        minority_info.append((cls_name, cnt, pct))
    print(f'{rank:<5} {str(cls_name):<25} {cnt:>10,} {pct:>11.4f}%  {status}')

print(separator)
majority_count = counts[0]
smallest_minority_name = names[-1]
smallest_minority_count = counts[-1]
imbalance_ratio = majority_count / max(1, smallest_minority_count)

print(f'\nQUANTITATIVE SUMMARY')
print(f'  Total Training Samples   : {total:,}')
print(f'  Total Classes            : {len(names)}')
print(f'  Majority Class           : "{names[0]}" ({majority_count:,} samples, {majority_count/total*100:.2f}%)')
print(f'  Rarest Minority Class    : "{smallest_minority_name}" ({smallest_minority_count:,} samples, {smallest_minority_count/total*100:.4f}%)')
print(f'  Severe Imbalance Ratio   : {imbalance_ratio:.1f} : 1')
print(f'  Underrepresented Attacks : {len(minority_info)} class(es)')
print()
print('RESEARCH JUSTIFICATION')
print(f'  The dataset exhibits an empirical imbalance ratio of {imbalance_ratio:.1f}:1.')
print('  Without augmentation, conventional classifiers are heavily biased toward majority traffic')
print('  and suffer severe detection failures on rare attack patterns.')
print('  This empirically justifies the requirement for CTGAN augmentation in Module 2.')
print(separator)

In [ ]:
# Chart 5 — Class Imbalance Bar Chart (Log Scale for Minority Visibility)
fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#2c3e7a' if i == 0 else '#c0392b' for i in range(len(names))]
bars = ax.bar(names, counts, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_yscale('log')

for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, val * 1.15,
            f'{val:,}\n({val/total*100:.2f}%)', ha='center', va='bottom', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2c3e7a', label=f'Majority Class: {names[0]}'),
    Patch(facecolor='#c0392b', label='Minority Attack Classes')
]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_title('Class Imbalance Report — GeNIS Training Set (Log Scale)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Traffic Class', fontsize=11)
ax.set_ylabel('Number of Samples (Log Scale)', fontsize=11)
ax.set_ylim(bottom=1, top=counts.max() * 5)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
chart5_path = os.path.join(BASE_DIR, 'chart5_class_imbalance_report.png')
plt.savefig(chart5_path, bbox_inches='tight')
plt.show()
print(f'Chart 5 saved: {chart5_path}')

---
## T1.8 — Baseline Benchmark Model (No Augmentation)

We train a Random Forest classifier on the **leak-free, unaugmented training set**
and evaluate on the **untouched test set**. With target leakage eliminated,
this produces authentic, honest performance metrics to benchmark against Module 2.

In [ ]:
print('Training baseline Random Forest classifier on unaugmented data...')
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)
y_pred = rf_baseline.predict(X_test)
print('Training complete. Evaluating on held-out test partition...')

In [ ]:
# Baseline Results — Empirical Classification Report
separator = '=' * 65
print(separator)
print('NETGUARDAI — BASELINE MODEL RESULTS (NO AUGMENTATION)')
print(f'Random Forest  |  GeNIS Test Set ({X_test.shape[0]:,} samples)  |  T1.8')
print(separator)
print()
print(classification_report(
    y_test, y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
))

macro_f1  = f1_score(y_test, y_pred, average='macro', zero_division=0)
macro_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f'Overall Macro F1-Score    : {macro_f1:.4f}')
print(f'Overall Macro Recall      : {macro_rec:.4f}')
print(f'Overall Weighted F1-Score : {weighted_f1:.4f}')
print()
print('FINDINGS')
print('  With target leakage eliminated, these metrics represent the true baseline performance.')
print('  Observe how performance drops significantly on minority attack categories.')
print('  This forms the exact empirical gap that CTGAN augmentation aims to solve in Module 2.')
print(separator)

In [ ]:
# Chart 6 — Per-Class F1 Score Chart
per_class_f1 = f1_score(y_test, y_pred, average=None, labels=np.arange(len(class_names)), zero_division=0)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#c0392b' if f < 0.50 else '#27ae60' for f in per_class_f1]
bars = ax.bar(class_names, per_class_f1, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axhline(y=0.50, color='gray', linestyle='--', linewidth=1, label='0.50 Detection Threshold')

for bar, val in zip(bars, per_class_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

legend_elements = [
    Patch(facecolor='#27ae60', label='F1 >= 0.50 (Acceptable)'),
    Patch(facecolor='#c0392b', label='F1 < 0.50 (Poor Detection)')
]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_title('Baseline Model — Per-Class F1 Score (No Augmentation)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Attack Class', fontsize=11)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_ylim(0, 1.15)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
chart6_path = os.path.join(BASE_DIR, 'chart6_baseline_f1_scores.png')
plt.savefig(chart6_path, bbox_inches='tight')
plt.show()
print(f'Chart 6 saved: {chart6_path}')

In [ ]:
# Chart 7 — Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(max(8, len(class_names)*1.5), max(6, len(class_names)*1.2)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title('Baseline Model — Confusion Matrix (No Augmentation)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
chart7_path = os.path.join(BASE_DIR, 'chart7_confusion_matrix.png')
plt.savefig(chart7_path, bbox_inches='tight')
plt.show()
print(f'Chart 7 saved: {chart7_path}')

---
## Module 1 — Final Summary

In [ ]:
print('=' * 65)
print('NETGUARDAI — MODULE 1 COMPLETE (LEAK-FREE BENCHMARK)')
print('=' * 65)
print()
print('Methodological Milestones Achieved:')
print('  [DONE] T1.1 — GeNIS dataset acquired from Zenodo Record 14919237')
print('  [DONE] T1.2 — Full 60s scenario coverage loaded (both train & test partitions)')
print('  [DONE] T1.3 — 4 EDA diagnostic charts generated')
print('  [DONE] T1.4 — Target isolation applied; Stratified split FIRST (70/15/15)')
print('  [DONE] T1.5 — Feature engineering fitted STRICTLY on Training split')
print('  [DONE] T1.6 — Normalization fitted STRICTLY on Training split')
print('  [DONE] T1.7 — Quantitative class imbalance report generated')
print('  [DONE] T1.8 — Honest Random Forest baseline trained without target leakage')
print()
print('Clean Partitions Ready for Module 2 Augmentation:')
print(f'  X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}    |  y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')
print()
print('Deliverables Saved:')
print('  chart1_class_distribution.png')
print('  chart2_missing_values.png (if applicable)')
print('  chart3_correlation_matrix.png')
print('  chart4_feature_distributions.png')
print('  chart5_class_imbalance_report.png')
print('  chart6_baseline_f1_scores.png')
print('  chart7_confusion_matrix.png')
print('=' * 65)